In [ ]:
import numpy as np
import cupy as cp
from scipy import ndimage
from types import SimpleNamespace
from scipy.fft import fftn, ifftn  # faster than np.fft on CPU

from holotomocupy.rec import Rec
from holotomocupy.shift import Shift
from holotomocupy.tomo_batched import TomoBatched
from holotomocupy.utils import *

# Acqusition parameters

In [ ]:
n = 128
ntheta = 360
detector_pixelsize = 1.4760147601476e-6 * 2 * 2048/n
energy = 17.1
wavelength = 1.24e-09 / energy
focustodetectordistance = 1.217

sx0 = -3.135e-3
z1 = np.array([5.110, 5.464, 6.879, 9.817, 10.372, 11.146, 12.594, 17.209]) * 1e-3 - sx0
z1_ids = np.array([0, 1, 2, 3]) ### note using index starting from 0
z1 = z1[z1_ids]
ndist = len(z1)
z2 = focustodetectordistance - z1
distances = (z1 * z2) / focustodetectordistance
magnifications = focustodetectordistance / z1
norm_magnifications = magnifications / magnifications[0]

nobj = int(np.ceil(n / norm_magnifications[-1] / 64)) * 64 
theta = cp.linspace(0,np.pi,ntheta).astype('float32')


# Read data

In [ ]:
data = np.load(f'data/data_{n}.npy')
ref = np.load(f'data/ref_{n}.npy')
pos = np.load(f'data/pos_{n}.npy')

# Shift data back accoridng to the random shifts, extend higher magnification data

In [ ]:
cl_shift = Shift(n, nobj,n,nobj, 1 / norm_magnifications)

distances_pag = distances / norm_magnifications**2
npad = n // 16
cref = cp.array(ref)

srdata_out = np.empty([ntheta, ndist, nobj, nobj], dtype='float32')
srdata = cp.zeros([ndist, nobj, nobj], dtype='float32')
r = pos
for j in range(ntheta):                        
    rdata = cp.array(data[j]) / (cref + 1e-5)
    if j%100==0:
        print(j)
        mshow_complex(rdata[0]+1j*rdata[ndist-1],True)

    for k in range(ndist - 1, -1, -1):
        tmp = rdata[k].astype('complex64')            
        tmp = cl_shift.Sback(tmp[None], cp.array(r[j:j+1, k]), k)[0].real
        padx0 = int((nobj - n / norm_magnifications[k]) / 2) - int(r[j,k,1])  
        pady0 = int((nobj - n / norm_magnifications[k]) / 2) - int(r[j,k,0])
        padx1 = int((nobj - n / norm_magnifications[k]) / 2) + int(r[j,k,1])  
        pady1 = int((nobj - n / norm_magnifications[k]) / 2) + int(r[j,k,0])
        padx0 = min(nobj,max(0,padx0))
        pady0 = min(nobj,max(0,pady0))
        padx1 = min(nobj,max(0,padx1))
        pady1 = min(nobj,max(0,pady1))
        padx0+=4
        pady0+=4
        padx1+=4
        pady1+=4
                
        tmp = cp.pad(tmp[pady0:-pady1], ((pady0,pady1),(0,0)),'edge')
        tmp = cp.pad(tmp[:,padx0:-padx1], ((0,0),(padx0,padx1)),'edge')
        # tmp = cp.pad(tmp[:,padx0:-padx1], ((0,0),(padx0,padx1)),'linear_ramp',end_values=((1, 1), (1, 1)) )        
        if k < ndist - 1:                    
            wx = cp.ones([nobj], dtype='float32')
            wy = cp.ones([nobj], dtype='float32')
            
            v = cp.linspace(0, 1, npad, endpoint=False)
            v = v**5 * (126 - 420 * v + 540 * v**2 - 315 * v**3 + 70 * v**4)
            wx[:padx0] = 0
            wx[padx0:padx0+npad] = v
            wx[-padx1-npad:-padx1] = 1 - v
            wx[-padx1:] = 0
            wy[:pady0] = 0
            wy[pady0:pady0+npad] = v
            wy[-pady1-npad:-pady1] = 1 - v
            wy[-pady1:] = 0
            
            w = cp.outer(wy, wx)
            tmp = tmp * w + srdata[k+1] * (1 - w)
        srdata[k] = tmp  
    
    for k in range(ndist):
        srdata_out[j,k] = cl_shift.coeffback(srdata[k],0).real.get()
    
    if j%100==0:
        print(j)
        mshow_complex(srdata_out[j,0]+1j*srdata_out[j,ndist-1],True)
                                    
srdata = srdata_out

# Multi-Paganin

In [ ]:
def multiPaganin(data, distances, wavelength, voxelsize, delta_beta, alpha):
    fx = cp.fft.fftfreq(data.shape[-1], d=voxelsize).astype('float32')
    [fx, fy] = cp.meshgrid(fx, fx)
    numerator = 0
    denominator = 0
    for j in range(data.shape[0]):        
        rad_freq = cp.fft.fft2(data[j])
        taylorExp = 1 + wavelength * distances[j] * cp.pi * delta_beta * (fx**2 + fy**2)
        numerator += taylorExp * rad_freq
        denominator += taylorExp**2

    numerator /= len(distances)
    denominator = (denominator / len(distances)) + alpha

    phase = cp.log(cp.real(cp.fft.ifft2(numerator / denominator)))
    phase *= delta_beta * 0.5

    return phase

def rec_init(rdata, paganin):
    recMultiPaganin = np.zeros([ntheta, nobj, nobj], dtype="float32")
    for j in range(ntheta):
        r = cp.array(rdata[j])
        mm = np.mean(r[:, :32 * n // 512]).get()
        r = cp.pad(r, ((0, 0), (nobj // 8, nobj // 8), (nobj // 8, nobj // 8)), 'constant', constant_values=mm)
        distances_pag = (distances / norm_magnifications**2)
        magnifications = focustodetectordistance / z1
        voxelsize = detector_pixelsize / magnifications[0]
        r = multiPaganin(r, distances_pag, wavelength, voxelsize, paganin, 1e-5)
        recMultiPaganin[j] = r[nobj // 8:-nobj // 8, nobj // 8:-nobj // 8].get()
        
    recMultiPaganin -= np.median(recMultiPaganin[:, :, :16 * n // 512])
    recMultiPaganin = recMultiPaganin + 1j * (recMultiPaganin / paganin).astype('float32')
    
    return recMultiPaganin

mask_r = 0.9
x = np.linspace(-1, 1, nobj)
[x, y] = np.meshgrid(x, x)
circ = (x**2 < mask_r).astype('float32')
g = np.exp(-30**2 * (x**2 + y**2))
fcirc = np.fft.fftshift(np.fft.fft2(np.fft.fftshift(circ)))
fg = np.fft.fftshift(np.fft.fft2(np.fft.fftshift(g)))
mask = np.fft.fftshift(np.fft.ifft2(np.fft.fftshift(fcirc * fg))).real.astype('float32')
mask /= np.amax(mask)
# mshow(mask, True)

psi_data = rec_init(srdata, 100)
psi_data *= mask    
mshow_complex(psi_data[-1], True)

# Tomo solver

In [ ]:
args = SimpleNamespace()
args.ngpus = cp.cuda.runtime.getDeviceCount()

args.ntheta = ntheta
args.nobj = nobj
args.nzobj = nobj

args.nchunk = 16
args.show = True
args.theta = theta
args.mask_r = 1.1

cl_rec = TomoBatched(args)
rec = cl_rec.rec_tomo(psi_data, 21)

mshow_complex(rec[rec.shape[0] // 2], args.show)
mshow_complex(rec[rec.shape[0] // 2, nobj // 2 - nobj // 8:nobj // 2 + nobj // 8, nobj // 2 - nobj // 8:nobj // 2 + nobj // 8], args.show)

# Save initial guess

In [ ]:
os.makedirs('rec', exist_ok=True)
np.save(f'rec/rec_init_{n}',rec)